In [ ]:
# -*- coding: utf-8 -*-
#
#  Copyright 2026 United Kingdom Research and Innovation
#
#  Licensed under the Apache License, Version 2.0 (the "License");
#  you may not use this file except in compliance with the License.
#  You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
#  Unless required by applicable law or agreed to in writing, software
#  distributed under the License is distributed on an "AS IS" BASIS,
#  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#  See the License for the specific language governing permissions and
#  limitations under the License.
#
#   Authored by:    Franck Vidal (UKRI-STFC)

![gVXR](https://github.com/TomographicImaging/gVXR-Tutorials/blob/main/img/Logo-transparent-small.png?raw=1)

# World coordinate system convention: right-handed

gVXR started as a computer graphics API dedicated to simulating X-ray images in VR applications.
The right-handed convention was chosen "by default". This is because it is implemented in OpenGL where the thumb points to
+X (right), the index finger to +Y (up), and the middle finger to +Z (towards the viewer/out of screen). Positive
rotations are counterclockwise about the axis of rotation.

See this excellent article by Ahmet Burul about
[Coordinate Systems of 3D Applications Guide](https://ahmetburul.medium.com/coordinate-systems-of-3d-applications-guide-ddfa2194ed88) for a short summary.

<div class="alert alert-block alert-warning">
    <b>Note:</b> Make sure the Python packages are already installed. See <a href="../README.md">README.md</a> in the root directory of the repository. If you are running this notebook from Google Colab, please run the cell below to install the required packages.
</div>

In [ ]:
from math import sqrt
import numpy as np
from matplotlib import pyplot as plt
from gvxrPython3 import gvxr

In [ ]:
source_position = [0, 0, -10]
detector_position = [0, 0, 50]

In [ ]:
# gvxr.createNewContext("OPENGL", True, -1, 4, 6, 32)
gvxr.createOpenGLContext();


The source is located at the position [0, 0, -10] cm. It corresponds to a cone beam geometry.

In [ ]:
gvxr.setSourcePosition(*source_position, "cm")
gvxr.usePointSource()

if gvxr.getMajorVersionOfSimpleGVXR() >= 2 and gvxr.getMinorVersionOfSimpleGVXR() >= 1:
    gvxr.setMonoChromaticPerPixelAtSDD(100, "keV", 15000)
else:
    gvxr.setMonoChromatic(100, "keV", 15000)

The detector is located at [0, 0, 50] cm. There are $640 \times 320$ pixels of $0.5 \times 0.5$ mm<sup>2</sup>. The
up vector is [0, -1, 0]. It is a unit vector that encodes the vertical axis of the detector, i.e. the translation vector
from the **first pixel of the first row** of the image to the **first pixel of the second row**.

By default, the horizontal axis, i.e. the translation vector
from the **first pixel of the first row** of the image to the **second pixel of the first row**, is calculated as
follows:

$$\mathbf{direction} = \mathbf{detector\_position} - \mathbf{source\_position}$$
$$\mathbf{horizontal\_axis} = \frac{\mathbf{direction}}{||\mathbf{direction}||} \times \mathbf{vertical\_axis}$$

It is possible to retrieve the value of the horizontal axis with `gvxr.getDetectorRightVector()`. In this case:

- $\mathbf{detector\_position} = [0, 0, 50]$
- $\mathbf{source\_position} = [0, 0, -10]$
- $\mathbf{direction} = [0, 0, 50] - [0, 0, -10] = [0, 0, 60]$
- $\frac{\mathbf{direction}}{||\mathbf{direction}||} = [0, 0, 1]$
- $\mathbf{horizontal\_axis} =  [0, 0, 1] \times [0, -1, 0] = [1, 0, 0]$

In [ ]:
vertical_axis = [0, -1, 0]
gvxr.setDetectorPosition(*detector_position, "cm")
gvxr.setDetectorUpVector(*vertical_axis)
gvxr.setDetectorNumberOfPixels(640, 320)
gvxr.setDetectorPixelSize(0.5, 0.5, "mm")

horizontal_axis = gvxr.getDetectorRightVector()
print("horizontal axis:", gvxr.getDetectorRightVector())

Create a pyramid. The base is a $4 \times 4$ cm<sup>2</sup> square. The height is 2 cm. It is made of silicon carbide.

In [ ]:
gvxr.removePolygonMeshesFromSceneGraph()

In [ ]:
vertices = [
    0, 2, 0,
    2, 0, -2,
    2, 0,  2,
    -2, 0, 2,
    -2, 0, -2,
]

indices = [
    0, 2, 1,
    0, 3, 2,
    0, 4, 3,
    0, 1, 4,
    1, 2, 3,
    1, 3, 4
]

gvxr.makeTriangularMesh("pyramid", vertices, indices, "cm")
gvxr.setCompound("pyramid", "SiC")
gvxr.setDensity("pyramid", 3.21, "g/cm3")
gvxr.addPolygonMeshAsOuterSurface("pyramid")

Add another object at the location [] cm.

In [ ]:
sphere_position = -5, 1, 0
gvxr.makeSphere("sphere", 10, 10, 1, "mm")
gvxr.translateNode("sphere", *sphere_position, "mm")
gvxr.addPolygonMeshAsInnerSurface("sphere")
gvxr.setElement("sphere", "Fe")

Compute the corresponding X-ray image

In [ ]:
xray_image1 = np.array(gvxr.computeXRayImage()) / gvxr.getTotalEnergyWithDetectorResponse()

Generate the corresponding 3D visualisation and take a screenshot

In [ ]:
gvxr.setWindowSize(800, 600)
gvxr.setZoom(650)
gvxr.setSceneRotationMatrix((-0.457769513130188, -0.12633083760738373, 0.8800499439239502, 0.0,
     0.0952025055885315, 0.9771969318389893, 0.18979720771312714, 0.0,
     -0.883959174156189, 0.17066653072834015, -0.43530359864234924, 0.0,
     0.0, 0.0, 0.0, 1.0))
gvxr.useWireframe()

gvxr.displayScene()

screenshot1 = np.array(gvxr.takeScreenshot())


We can change the orientation of the detector with `gvxr.setDetectorUpVector()` and `gvxr.setDetectorRightVector()`.

In [ ]:
vertical_axis2 = [0, 1, 0]
horizontal_axis2 = [-1, 0, 0]
gvxr.setDetectorUpVector(*vertical_axis2)
gvxr.setDetectorRightVector(*horizontal_axis2)

Update the X-ray image and the 3D visualisation.

In [ ]:
xray_image2 = np.array(gvxr.computeXRayImage()) / gvxr.getTotalEnergyWithDetectorResponse()

gvxr.displayScene()
screenshot2 = np.array(gvxr.takeScreenshot())

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(15, 15))

axs[0, 0].set_title("vertical axis: [" + " ".join([str(item) for item in vertical_axis]) + "]\nhorizontal axis: [" +
                    " ".join([str(item) for item in horizontal_axis]) + "]")
axs[0, 0].imshow(-np.log(xray_image1), cmap="gray")
axs[0, 1].imshow(screenshot1)

axs[1, 0].set_title("vertical axis: [" + " ".join([str(item) for item in vertical_axis2]) + "]\nhorizontal axis: [" +
                    " ".join([str(item) for item in horizontal_axis2]) + "]")
axs[1, 0].imshow(-np.log(xray_image2), cmap="gray")
axs[1, 1].imshow(screenshot2)


Uncomment the cell below to use the interactive 3D visualisation.

- It is only available if windowing is enable (e.g.
using the `"OPENGL"` backend)
- The user can rotate the 3D scene and zoom-in and -out in the visualisation window.
- Keys are:
    - Q/Escape: to quit the event loop (does not close the window)
    - B: display/hide the X-ray beam
    - W: display the polygon meshes in solid or wireframe
    - N: display the X-ray image in negative or positive
    - H: display/hide the X-ray detector
- Mouse interactions:
    - Zoom in/out: mouse wheel
    - Rotation: Right mouse button down + move cursor
- To activate the interactive visualisation mode call:
  ```python
  gvxr.renderLoop()
  ```
  Note that the call stops the execution of other functions. You must 1st exit the interactive visualisation mode before calling further Python functions.
- To exit the interactive visualisation mode press `<Q>` or `<ESC>`, or close the window.

In [ ]:
# gvxr.renderLoop()

Change the viewing angle

In [ ]:
gvxr.setSceneRotationMatrix((-0.9495478868484497, -0.025659190490841866, 0.3125735819339752, 0.0,
    0.04726913943886757, 0.9735544919967651, 0.22351303696632385, 0.0,
    -0.3100402057170868, 0.22701004147529602, -0.9232224822044373, 0.0,
    0.0, 0.0, 0.0, 1.0))

gvxr.setZoom(460)

Update the screenshot

In [ ]:
xray_image3 = np.array(gvxr.computeXRayImage()) / gvxr.getTotalEnergyWithDetectorResponse()

gvxr.displayScene()
screenshot3 = np.array(gvxr.takeScreenshot())

Rotate the sphere around the Z-axis.

In [ ]:
gvxr.setNodeTransformationMatrix("sphere", [[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])
gvxr.rotateNode("sphere", 90, 0, 0, 1)
gvxr.translateNode("sphere", *sphere_position, "mm")

xray_image4 = np.array(gvxr.computeXRayImage()) / gvxr.getTotalEnergyWithDetectorResponse()

gvxr.displayScene()
screenshot4 = np.array(gvxr.takeScreenshot())

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(15, 15))

axs[0, 0].set_title("Rotation angle: 0\nRotation axis: [0, 0, 1]")
axs[0, 0].imshow(-np.log(xray_image3), cmap="gray")
axs[0, 1].imshow(screenshot3)

axs[1, 0].set_title("Rotation angle: 90\nRotation axis: [0, 0, 1]")
axs[1, 0].imshow(-np.log(xray_image4), cmap="gray")
axs[1, 1].imshow(screenshot4)


In [ ]:
# gvxr.renderLoop()